In [1]:
from gradio import networking
networking.url_ok = lambda url: True


In [2]:
import sys, gradio
print("Python:", sys.executable)
print("Gradio:", gradio.__version__)

Python: /Users/saiashish/cinema-match/.venv/bin/python
Gradio: 4.44.1


# 🎬 Cinema Match v2 — Now With Script Upload

**What's new in v2:**
- 📄 Upload screenplays (PDF, DOCX, TXT, FOUNTAIN, MD) — universal parser
- 🎯 Auto-extract scenes, characters, structure
- 🧠 7th agent: **Script Analyst** for arc/pacing/character questions
- 🔄 All existing agents become **script-aware** (pull relevant scenes when answering)
- 🔗 **Shareable URLs** — every uploaded script gets a permanent link
- 💾 SQLite persistence — scripts survive restarts

**Architecture:**

```
┌─────────────────────────────────────────────────────────┐
│  User: uploads script.pdf + asks question               │
└─────────────────────────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────┐
│  Parser: PDF/DOCX/TXT/FOUNTAIN/MD → plain text          │
│  Scene Extractor: regex → list of scenes + characters   │
│  Storage: SQLite (script_id, content, scenes, ...)      │
│  Embedder: chunks → vector store (per-script index)     │
└─────────────────────────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────┐
│  LangGraph Router (intent + script_loaded?)             │
└─────────────────────────────────────────────────────────┘
                     │
   ┌──────┬──────┬───┴──┬───────┬───────┬───────┐
   ▼      ▼      ▼      ▼       ▼       ▼       ▼
 Script Music Camera Tutor  ShotIdea Network  🆕 Script
 Agent  Agent Agent  Agent  Agent    Agent    Analyst
   └──────┴──────┴──────┴───┬───┴───────┴───────┘
                            │
                            ▼ (each agent retrieves relevant scenes)
                       Script context
                            │
                            ▼
                       Final answer
```

`MOCK_MODE = True` by default — runs end-to-end with no API key.


## 1. Setup & Installation

**New v2 dependencies:** `pypdf` (PDF parsing), `python-docx` (Word parsing).

The notebook gracefully degrades — if a parser is missing, that format just gets disabled.

In [3]:
# Run once on first setup. After that, comment out.
import sys, subprocess

PACKAGES = [
    "anthropic>=0.40.0",
    "langgraph>=0.2.0",
    "langchain-core>=0.3.0",
    "sentence-transformers>=2.5.0",
    "faiss-cpu>=1.7.4",
    "gradio>=4.31.0",
    "pypdf>=4.0.0",            # NEW: PDF parsing
    "python-docx>=1.1.0",      # NEW: DOCX parsing
    "numpy",
    "pandas",
]

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# Uncomment on first run:
# pip_install(PACKAGES)
print("✅ Setup cell ready")


✅ Setup cell ready


## 2. Configuration

Plus a new `STORAGE_DIR` for SQLite + uploaded files.

In [4]:
import os
from dataclasses import dataclass
from pathlib import Path

@dataclass
class Config:
    MOCK_MODE: bool = True
    MODEL: str = "claude-sonnet-4-5"
    EMBED_MODEL: str = "sentence-transformers/all-MiniLM-L6-v2"
    MAX_TOKENS: int = 2048           # bumped — script analysis needs more headroom
    TEMPERATURE: float = 0.7
    ANTHROPIC_API_KEY: str = os.getenv("ANTHROPIC_API_KEY", "")
    STORAGE_DIR: str = "./cinema_match_data"
    DB_PATH: str = "./cinema_match_data/cinema_match.db"
    BASE_URL: str = "http://localhost:7860"   # used to construct share URLs

cfg = Config()
Path(cfg.STORAGE_DIR).mkdir(exist_ok=True)
print(f"MOCK_MODE   = {cfg.MOCK_MODE}")
print(f"Storage     = {cfg.STORAGE_DIR}")
print(f"Database    = {cfg.DB_PATH}")
print(f"API key     = {'set' if cfg.ANTHROPIC_API_KEY else 'NOT set (mock mode required)'}")


MOCK_MODE   = True
Storage     = ./cinema_match_data
Database    = ./cinema_match_data/cinema_match.db
API key     = NOT set (mock mode required)


## 3. LLM Wrapper

Same as v1, plus mock responses for the new Script Analyst agent.

In [5]:
from typing import Optional

MOCK_RESPONSES = {
    "script": """SCENE 4 — INT. ABANDONED LIGHTHOUSE — NIGHT

MAYA (32, weathered, eyes scanning the storm) climbs the spiral stairs.
The wind howls through broken windows.

MAYA (V.O.)
Some places remember things people try to forget.

She reaches the top. A single photograph waits on the lamp's base.""",
    "music": """Recommended tracks for this scene:

1. **'Time' — Hans Zimmer** — slow build, emotional weight (BPM 65)
2. **'An Ending (Ascent)' — Brian Eno** — ambient, contemplative
3. **'The Wolf' — SIAMÉS** — tense electronic (BPM 95)

Top pick: 'Time' — its slow piano-to-orchestral build matches a revelation scene.""",
    "camera": """**Primary movement: Slow dolly-in (push-in)**
- Starts wide, tightens on Maya's face as she reads
- Reference: Spielberg's 'Jaws'

**Coverage:**
- Wide establishing shot (drone or crane)
- Over-the-shoulder as she finds the photo
- Extreme close-up on the photograph

**Lens:** 35mm anamorphic for the wide; 85mm for OTS.""",
    "tutor": """**How to compose this revelation shot:**

1. Block the actor first — Rembrandt lighting (3/4 face lit)
2. Lock camera on sticks — handheld kills stillness
3. Frame using rule-of-thirds — eyes on upper third
4. Shallow depth of field (f/2.0–2.8) — isolate the photograph
5. Time the dolly to her breath — 4-second push on inhale
6. Hold the final frame 2 extra seconds post-line

**Common mistake:** cutting too soon. Resist it.""",
    "shot_ideas": """6 angle ideas:

1. **Low-angle OTS** — Maya looms; the photo dwarfs her
2. **Top-down 'God shot'** — vulnerability
3. **Dutch tilt close-up** — disorientation, worldview cracking
4. **Mirror reflection in lamp glass** — duality
5. **POV through window from outside** — voyeuristic distance
6. **Macro insert on photograph's edge** — pull focus to face

Pick #1 if defiant, #3 if losing control, #5 for emotional remove.""",
    "network": """Top 3 matches:

1. **Priya Raman** — Producer, $500K-$2M indie thrillers. Score: 0.91
2. **Marcus Vela** — Cinematographer, coastal/atmospheric horror. Score: 0.87
3. **Lena Okafor** — Composer, Sundance 2023. Score: 0.84

Reach out to Priya first.""",
    "script_analyst": """**Script Analysis: Maya's Arc**

**Want vs. Need:**
- Want: To uncover what her father hid
- Need: To accept that some truths can't be fully known

**Three-act structure:**
- Act 1 (pp 1-25): Returns home after father's death; finds first clue
- Act 2 (pp 26-75): Investigation deepens; lighthouse becomes obsession
- Act 3 (pp 76-95): Confronts the truth; lets go

**Pacing notes:**
- Strong opening hook (page 3)
- Sag in middle of Act 2 (pp 45-58) — consider compressing
- Climax lands cleanly

**Character voice:** Maya is consistently terse, observation-heavy. Strong.

**Top 3 revisions to consider:**
1. Tighten Act 2 midpoint
2. Add 1-2 lines clarifying Maya's relationship with her sister
3. The lighthouse photograph reveal could come 5 pages earlier""",
    "default": "I'm Cinema Match. Upload a script or ask me about scriptwriting, music, camera angles, shot composition, or finding collaborators."
}

def _detect_agent(prompt: str) -> str:
    p = prompt.lower()
    if any(k in p for k in ["analyze", "arc", "pacing", "structure", "critique"]): return "script_analyst"
    if any(k in p for k in ["script", "scene", "dialogue", "story"]): return "script"
    if any(k in p for k in ["music", "song", "track", "score"]): return "music"
    if any(k in p for k in ["camera movement", "dolly", "tracking"]): return "camera"
    if any(k in p for k in ["how to", "teach", "tutorial"]): return "tutor"
    if any(k in p for k in ["angle", "shot idea", "framing"]): return "shot_ideas"
    if any(k in p for k in ["producer", "composer", "find", "match"]): return "network"
    return "default"

def call_llm(prompt: str, system: Optional[str] = None, agent_hint: Optional[str] = None) -> str:
    if cfg.MOCK_MODE:
        key = agent_hint or _detect_agent(prompt)
        return MOCK_RESPONSES.get(key, MOCK_RESPONSES["default"])

    from anthropic import Anthropic
    client = Anthropic(api_key=cfg.ANTHROPIC_API_KEY)
    resp = client.messages.create(
        model=cfg.MODEL,
        max_tokens=cfg.MAX_TOKENS,
        temperature=cfg.TEMPERATURE,
        system=system or "You are a helpful film-industry expert.",
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.content[0].text

print(call_llm("Analyze the protagonist's arc")[:200])


**Script Analysis: Maya's Arc**

**Want vs. Need:**
- Want: To uncover what her father hid
- Need: To accept that some truths can't be fully known

**Three-act structure:**
- Act 1 (pp 1-25): Returns 


## 4. Knowledge Bases (cinematography, music, network)

Same as v1 — small but illustrative. In production these grow to thousands of entries.

In [6]:
CINEMATOGRAPHY_KB = [
    {"id": "cam_001", "topic": "dolly_in", "text": "A dolly-in slowly pushes the camera toward the subject, building emotional intensity. Used by Spielberg, Kubrick. Best for revelation, decision, or recognition moments."},
    {"id": "cam_002", "topic": "low_angle", "text": "Low-angle shot positions camera below subject eye-line, making them appear powerful, dominant, or threatening."},
    {"id": "cam_003", "topic": "high_angle", "text": "High-angle shot looks down on the subject, conveying vulnerability, weakness, or insignificance."},
    {"id": "cam_004", "topic": "dutch_tilt", "text": "Dutch tilt rotates the camera off horizontal axis, creating unease, disorientation, or psychological imbalance."},
    {"id": "cam_005", "topic": "tracking_shot", "text": "Tracking shot follows a subject in motion, immersing the audience. Iconic in Goodfellas, 1917."},
    {"id": "cam_006", "topic": "rack_focus", "text": "Rack focus shifts focus from one plane to another within a single shot."},
    {"id": "cam_007", "topic": "ots_shot", "text": "Over-the-shoulder shot is the workhorse of dialogue scenes."},
    {"id": "cam_008", "topic": "rule_of_thirds", "text": "Rule of thirds: divide frame into 3x3 grid; place subject eyes on upper-third line."},
    {"id": "cam_009", "topic": "lens_choice", "text": "Wide lenses (24-35mm) exaggerate space. 50mm is normal eye perspective. 85mm+ compresses depth, isolates subject."},
    {"id": "cam_010", "topic": "lighting_three_point", "text": "Three-point lighting: key, fill, back. Foundation of all studio lighting."},
    {"id": "cam_011", "topic": "rembrandt_lighting", "text": "Rembrandt lighting: triangle of light on cheek opposite the key light."},
    {"id": "cam_012", "topic": "handheld", "text": "Handheld camera adds urgency, realism, intimacy. Used in Bourne films, Children of Men."},
    {"id": "cam_013", "topic": "crane_shot", "text": "Crane shot moves the camera vertically/sweepingly through space."},
    {"id": "cam_014", "topic": "long_take", "text": "Long take (oner) holds without cutting. Examples: Birdman, 1917."},
]

MUSIC_KB = [
    {"id": "mus_001", "title": "Time", "artist": "Hans Zimmer", "mood": "emotional, building, contemplative", "tempo": 65, "use_for": "revelation, climax"},
    {"id": "mus_002", "title": "An Ending (Ascent)", "artist": "Brian Eno", "mood": "ambient, transcendent", "tempo": 60, "use_for": "endings, contemplation"},
    {"id": "mus_003", "title": "The Wolf", "artist": "SIAMÉS", "mood": "tense, electronic", "tempo": 95, "use_for": "thriller, chase"},
    {"id": "mus_004", "title": "Lux Aeterna", "artist": "Clint Mansell", "mood": "tense, dramatic", "tempo": 110, "use_for": "high tension"},
    {"id": "mus_005", "title": "Adagio for Strings", "artist": "Samuel Barber", "mood": "devastating, mournful", "tempo": 50, "use_for": "loss, grief"},
    {"id": "mus_006", "title": "Mind Heist", "artist": "Zack Hemsey", "mood": "epic, building", "tempo": 85, "use_for": "trailers, reveals"},
    {"id": "mus_007", "title": "Spiegel im Spiegel", "artist": "Arvo Pärt", "mood": "minimal, sacred", "tempo": 50, "use_for": "intimacy, fragility"},
    {"id": "mus_008", "title": "Genesis", "artist": "Justice", "mood": "energetic, electronic", "tempo": 120, "use_for": "openings, montages"},
]

NETWORK_KB = [
    {"id": "p_001", "name": "Priya Raman", "role": "Producer", "specialty": "indie thriller, $500K-$2M, festival circuit", "credits": "4 TIFF, 2 Sundance"},
    {"id": "p_002", "name": "Marcus Vela", "role": "Cinematographer", "specialty": "coastal atmospheric horror", "credits": "Coastal Echoes, The Tide"},
    {"id": "p_003", "name": "Lena Okafor", "role": "Composer", "specialty": "minimalist orchestral, ambient hybrid", "credits": "Coastal Echoes (Sundance 2023)"},
    {"id": "p_004", "name": "David Chen", "role": "Producer", "specialty": "documentaries, social-issue features", "credits": "3 Emmy noms"},
    {"id": "p_005", "name": "Sofia Marquez", "role": "Director of Photography", "specialty": "handheld vérité, dialogue-heavy drama", "credits": "12 features"},
    {"id": "p_006", "name": "Aiden Park", "role": "Composer", "specialty": "electronic, modern thrillers", "credits": "2 Netflix originals"},
    {"id": "p_007", "name": "Rachel Kim", "role": "Producer", "specialty": "horror, $1M-$5M, streaming-first", "credits": "8 horror features"},
]

print(f"📚 KBs loaded: cinema={len(CINEMATOGRAPHY_KB)}, music={len(MUSIC_KB)}, network={len(NETWORK_KB)}")


📚 KBs loaded: cinema=14, music=8, network=7


## 5. Vector Store

Same `VectorStore` class from v1. Graceful fallback to keyword search.

In [7]:
import numpy as np

class VectorStore:
    """Lightweight FAISS-backed semantic search over a list of dicts."""

    def __init__(self, items, text_fn, embed_model_name=None):
        self.items = items
        self.text_fn = text_fn
        self._embeddings = None
        self._index = None
        self._model = None
        self._embed_model_name = embed_model_name or cfg.EMBED_MODEL

    def _load_model(self):
        if self._model is None:
            try:
                from sentence_transformers import SentenceTransformer
                self._model = SentenceTransformer(self._embed_model_name)
            except Exception as e:
                print(f"⚠️  Embedding model unavailable ({e}); falling back to keyword search")
                self._model = "keyword"
        return self._model

    def build(self):
        model = self._load_model()
        texts = [self.text_fn(it) for it in self.items]
        if model == "keyword":
            self._embeddings = texts
            return
        self._embeddings = model.encode(texts, normalize_embeddings=True)
        try:
            import faiss
            d = self._embeddings.shape[1]
            self._index = faiss.IndexFlatIP(d)
            self._index.add(self._embeddings.astype("float32"))
        except Exception:
            self._index = None

    def search(self, query, k=3):
        if self._embeddings is None or len(self.items) == 0:
            if len(self.items) == 0:
                return []
            self.build()
        model = self._model
        if model == "keyword":
            q = query.lower()
            scored = [(sum(1 for w in q.split() if w in t.lower()), i) for i, t in enumerate(self._embeddings)]
            scored.sort(reverse=True)
            return [(self.items[i], float(s)) for s, i in scored[:k] if s > 0] or [(self.items[i], 0.0) for _, i in scored[:k]]
        q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
        if self._index is not None:
            scores, idxs = self._index.search(q_emb, k)
            return [(self.items[i], float(s)) for i, s in zip(idxs[0], scores[0])]
        sims = (self._embeddings @ q_emb[0])
        top = np.argsort(-sims)[:k]
        return [(self.items[i], float(sims[i])) for i in top]


camera_store  = VectorStore(CINEMATOGRAPHY_KB, lambda x: f"{x['topic']}: {x['text']}")
music_store   = VectorStore(MUSIC_KB,          lambda x: f"{x['title']} by {x['artist']}. Mood: {x['mood']}. Use for: {x['use_for']}")
network_store = VectorStore(NETWORK_KB,        lambda x: f"{x['role']}: {x['specialty']}. Credits: {x['credits']}")

print("🔨 Building vector stores...")
camera_store.build()
music_store.build()
network_store.build()
print("✅ Stores indexed")


🔨 Building vector stores...
✅ Stores indexed


## 6. 🆕 Universal File Parser

Handles **PDF, DOCX, TXT, FOUNTAIN, MD** through one interface. Each parser is wrapped in try/except so missing dependencies degrade gracefully.

**Why one interface?** The rest of the system never has to know what file format was uploaded — it just gets text back.

In [8]:
from pathlib import Path

def parse_pdf(path: str) -> str:
    """Extract text from a PDF file."""
    try:
        from pypdf import PdfReader
        reader = PdfReader(path)
        return "\n\n".join(page.extract_text() or "" for page in reader.pages)
    except ImportError:
        raise RuntimeError("pypdf not installed. Run: pip install pypdf")

def parse_docx(path: str) -> str:
    """Extract text from a .docx file."""
    try:
        from docx import Document
        doc = Document(path)
        return "\n".join(p.text for p in doc.paragraphs)
    except ImportError:
        raise RuntimeError("python-docx not installed. Run: pip install python-docx")

def parse_txt(path: str) -> str:
    """Read a plain text file (also handles .md and .fountain)."""
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

PARSERS = {
    ".pdf":      parse_pdf,
    ".docx":     parse_docx,
    ".txt":      parse_txt,
    ".md":       parse_txt,
    ".fountain": parse_txt,
}

def parse_script(path: str) -> str:
    """Universal entry point — sniff extension, dispatch to right parser."""
    ext = Path(path).suffix.lower()
    if ext not in PARSERS:
        raise ValueError(f"Unsupported format: {ext}. Supported: {list(PARSERS.keys())}")
    text = PARSERS[ext](path)
    if not text or len(text.strip()) < 50:
        raise ValueError(f"Parsed text suspiciously short ({len(text)} chars). File may be corrupt or scanned.")
    return text

# Quick smoke test with an inline sample
sample_screenplay = """
TITLE: THE LIGHTHOUSE KEEPER

FADE IN:

INT. ABANDONED LIGHTHOUSE - NIGHT

MAYA (32, weathered) climbs the spiral stairs. Wind howls through broken windows.

MAYA
Some places remember things people try to forget.

She reaches the top. A photograph rests on the lamp's base.

EXT. CLIFFTOP - CONTINUOUS

The lighthouse beam suddenly flickers to life. Below, the ocean churns.

MAYA (V.O.)
He always said the light would find me.

CUT TO:

INT. MAYA'S APARTMENT - DAY

Maya stares at the photograph. Her sister ELENA (38) enters.

ELENA
You shouldn't have gone back there.

FADE OUT.
"""

# Write to a temp file and round-trip through parse_script
import tempfile
with tempfile.NamedTemporaryFile(suffix=".txt", delete=False, mode="w") as f:
    f.write(sample_screenplay)
    temp_path = f.name

parsed = parse_script(temp_path)
print(f"✅ Parsed {len(parsed)} chars")
print(parsed[:200] + "...")


✅ Parsed 583 chars

TITLE: THE LIGHTHOUSE KEEPER

FADE IN:

INT. ABANDONED LIGHTHOUSE - NIGHT

MAYA (32, weathered) climbs the spiral stairs. Wind howls through broken windows.

MAYA
Some places remember things people t...


## 7. 🆕 Scene & Character Extractor

Screenplays follow a strict format. Scene headers always start with `INT.` or `EXT.` Character names are ALL CAPS lines before dialogue. Regex handles 95% of cases.

In [9]:
import re
from dataclasses import dataclass, field
from typing import List

@dataclass
class Scene:
    number: int
    heading: str          # "INT. ABANDONED LIGHTHOUSE - NIGHT"
    location: str         # "ABANDONED LIGHTHOUSE"
    time_of_day: str      # "NIGHT"
    int_ext: str          # "INT" or "EXT"
    content: str          # full scene text
    characters: List[str] = field(default_factory=list)

# Scene heading regex: INT./EXT. LOCATION - TIME
SCENE_HEADER_RE = re.compile(
    r"^\s*(INT\.|EXT\.|INT/EXT\.|I/E\.)\s*(.+?)(?:\s*[-–]\s*([A-Z\s]+))?\s*$",
    re.MULTILINE
)

# Character cue: ALL-CAPS line, 2-30 chars, possibly with (V.O.) or (O.S.)
CHARACTER_RE = re.compile(
    r"^\s*([A-Z][A-Z\s\.\-']{1,28}[A-Z])(?:\s*\([^)]+\))?\s*$",
    re.MULTILINE
)

# Words that look like character names but aren't
NON_CHARACTER_WORDS = {
    "FADE", "CUT", "DISSOLVE", "SMASH", "MATCH", "TITLE", "INT", "EXT",
    "CONTINUOUS", "LATER", "MOMENTS", "DAY", "NIGHT", "MORNING", "EVENING",
    "BLACK", "END", "TO", "THE", "THE END", "FADE IN", "FADE OUT", "CUT TO",
}

def extract_scenes(script_text: str) -> List[Scene]:
    """Parse a screenplay into a list of Scene objects."""
    matches = list(SCENE_HEADER_RE.finditer(script_text))
    scenes = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(script_text)
        content = script_text[start:end].strip()

        int_ext = m.group(1).rstrip(".")
        rest = m.group(2).strip()
        time_of_day = (m.group(3) or "").strip()

        # If rest still has " - X" attached, split it
        if " - " in rest and not time_of_day:
            location, time_of_day = rest.rsplit(" - ", 1)
        else:
            location = rest

        scene = Scene(
            number=i+1,
            heading=m.group(0).strip(),
            location=location.strip(),
            time_of_day=time_of_day.strip(),
            int_ext=int_ext,
            content=content,
        )
        scene.characters = extract_characters(content)
        scenes.append(scene)
    return scenes

def extract_characters(scene_text: str) -> List[str]:
    """Find character cues in a single scene."""
    candidates = CHARACTER_RE.findall(scene_text)
    chars = set()
    for c in candidates:
        c_clean = c.strip()
        # Filter: not a known non-character keyword, has 2+ letters, not all common words
        if c_clean in NON_CHARACTER_WORDS: continue
        if len(c_clean) < 2: continue
        if c_clean.startswith(("INT", "EXT", "FADE", "CUT", "DISSOLVE")): continue
        chars.add(c_clean)
    return sorted(chars)

def extract_all_characters(scenes: List[Scene]) -> List[str]:
    """Aggregate unique characters across all scenes."""
    all_chars = set()
    for s in scenes:
        all_chars.update(s.characters)
    return sorted(all_chars)

# Smoke test
scenes = extract_scenes(parsed)
all_chars = extract_all_characters(scenes)
print(f"✅ Extracted {len(scenes)} scenes")
print(f"✅ Found characters: {all_chars}")
for s in scenes:
    print(f"   Scene {s.number}: {s.heading}  → chars: {s.characters}")


✅ Extracted 3 scenes
✅ Found characters: ['ELENA', 'MAYA']
   Scene 1: INT. ABANDONED LIGHTHOUSE - NIGHT  → chars: ['MAYA']
   Scene 2: EXT. CLIFFTOP - CONTINUOUS  → chars: ['MAYA']
   Scene 3: INT. MAYA'S APARTMENT - DAY  → chars: ['ELENA']


## 8. 🆕 Script Storage (SQLite)

Persists uploaded scripts so they survive restarts and can be shared by URL.

**Schema:**
- `scripts` — id, title, content, scene_count, character_count, uploaded_at
- `scenes` — script_id, number, heading, content (for fast scene lookup)

**Why SQLite:** zero setup, single file, swap to Postgres later by changing the connection string.

In [10]:
import sqlite3
import secrets
import json
from datetime import datetime
from contextlib import contextmanager

@contextmanager
def db_conn():
    conn = sqlite3.connect(cfg.DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        yield conn
        conn.commit()
    finally:
        conn.close()

def init_db():
    """Create tables if they don't exist."""
    with db_conn() as conn:
        conn.executescript("""
        CREATE TABLE IF NOT EXISTS scripts (
            id              TEXT PRIMARY KEY,
            title           TEXT,
            content         TEXT NOT NULL,
            scene_count     INTEGER,
            character_count INTEGER,
            uploaded_at     TEXT
        );
        CREATE TABLE IF NOT EXISTS scenes (
            script_id   TEXT,
            number      INTEGER,
            heading     TEXT,
            content     TEXT,
            characters  TEXT,
            PRIMARY KEY (script_id, number),
            FOREIGN KEY (script_id) REFERENCES scripts(id)
        );
        """)

def save_script(content: str, scenes: List[Scene], title: Optional[str] = None) -> str:
    """Persist a script + its scenes; return the share-link ID."""
    script_id = secrets.token_urlsafe(8)   # ~11 chars, hard to guess
    all_chars = extract_all_characters(scenes)
    title = title or (scenes[0].heading[:60] if scenes else "Untitled")

    with db_conn() as conn:
        conn.execute(
            "INSERT INTO scripts (id, title, content, scene_count, character_count, uploaded_at) VALUES (?, ?, ?, ?, ?, ?)",
            (script_id, title, content, len(scenes), len(all_chars), datetime.utcnow().isoformat())
        )
        for s in scenes:
            conn.execute(
                "INSERT INTO scenes (script_id, number, heading, content, characters) VALUES (?, ?, ?, ?, ?)",
                (script_id, s.number, s.heading, s.content, json.dumps(s.characters))
            )
    return script_id

def load_script(script_id: str) -> Optional[dict]:
    """Load a script + its scenes by ID."""
    with db_conn() as conn:
        script = conn.execute("SELECT * FROM scripts WHERE id = ?", (script_id,)).fetchone()
        if not script:
            return None
        scenes_rows = conn.execute(
            "SELECT * FROM scenes WHERE script_id = ? ORDER BY number", (script_id,)
        ).fetchall()
        scenes = [
            Scene(
                number=r["number"], heading=r["heading"], content=r["content"],
                characters=json.loads(r["characters"]),
                location="", time_of_day="", int_ext="",
            )
            for r in scenes_rows
        ]
        return {
            "id": script["id"], "title": script["title"], "content": script["content"],
            "scene_count": script["scene_count"], "character_count": script["character_count"],
            "uploaded_at": script["uploaded_at"], "scenes": scenes,
        }

def list_scripts(limit: int = 20) -> List[dict]:
    with db_conn() as conn:
        rows = conn.execute(
            "SELECT id, title, scene_count, character_count, uploaded_at FROM scripts ORDER BY uploaded_at DESC LIMIT ?",
            (limit,)
        ).fetchall()
        return [dict(r) for r in rows]

def get_share_url(script_id: str) -> str:
    return f"{cfg.BASE_URL}/?script={script_id}"

# Initialize + smoke test
init_db()
test_id = save_script(parsed, scenes, title="The Lighthouse Keeper (test)")
print(f"✅ Saved script ID: {test_id}")
print(f"🔗 Share URL: {get_share_url(test_id)}")

loaded = load_script(test_id)
print(f"✅ Loaded back: '{loaded['title']}' — {loaded['scene_count']} scenes, {loaded['character_count']} chars")


✅ Saved script ID: FZgsWT1vLMs
🔗 Share URL: http://localhost:7860/?script=FZgsWT1vLMs
✅ Loaded back: 'The Lighthouse Keeper (test)' — 3 scenes, 2 chars


## 9. 🆕 Per-Script Scene Index

When a script is loaded, build a vector store **of its scenes**. This lets agents pull the right scene for any query — "suggest music for the scene where Maya finds the photo" → retrieves the lighthouse scene.

**Key design:** indexes are cached per-script-ID. If the user reloads the same script, we don't rebuild.

In [11]:
_script_indexes = {}   # script_id -> VectorStore

def get_script_index(script_id: str) -> Optional[VectorStore]:
    """Build (or fetch cached) scene-level vector store for a script."""
    if script_id in _script_indexes:
        return _script_indexes[script_id]

    data = load_script(script_id)
    if not data or not data["scenes"]:
        return None

    scene_dicts = [
        {
            "number": s.number, "heading": s.heading,
            "content": s.content, "characters": s.characters,
        }
        for s in data["scenes"]
    ]
    store = VectorStore(
        scene_dicts,
        lambda x: f"Scene {x['number']}: {x['heading']}\nCharacters: {', '.join(x['characters'])}\n{x['content'][:500]}"
    )
    store.build()
    _script_indexes[script_id] = store
    return store

# Smoke test
idx = get_script_index(test_id)
results = idx.search("revelation moment with photograph", k=2)
print(f"🔍 Top scene matches for 'revelation with photograph':")
for scene, score in results:
    print(f"   [{score:.2f}] Scene {scene['number']}: {scene['heading']}")


🔍 Top scene matches for 'revelation with photograph':
   [0.27] Scene 3: INT. MAYA'S APARTMENT - DAY
   [0.23] Scene 2: EXT. CLIFFTOP - CONTINUOUS


## 10. 🆕 Script-Aware Agent Wrapper

This is the key abstraction: a function that takes any agent and **automatically injects relevant scenes** when a script is loaded.

**Without this:** every agent has duplicated retrieval code.
**With this:** agents stay clean, script-awareness is one wrapper.

In [12]:
def with_script_context(agent_fn, agent_hint: str):
    """Decorator-like wrapper: adds script-context retrieval to any agent."""
    def wrapped(query: str, script_id: Optional[str] = None, k_scenes: int = 2) -> str:
        if not script_id:
            return agent_fn(query)   # no script loaded — original behavior

        idx = get_script_index(script_id)
        if not idx:
            return agent_fn(query)

        # Retrieve relevant scenes
        relevant = idx.search(query, k=k_scenes)
        scenes_text = "\n\n".join([
            f"--- Scene {s['number']}: {s['heading']} ---\n{s['content'][:800]}"
            for s, _ in relevant
        ])

        # Inject into the query so the agent sees both
        augmented_query = f"""[CONTEXT FROM UPLOADED SCRIPT]
{scenes_text}

[DIRECTOR'S QUESTION]
{query}

Answer with reference to the specific scenes above when relevant."""
        return agent_fn(augmented_query)
    return wrapped

print("✅ Script-context wrapper defined")


✅ Script-context wrapper defined


## 11. 🆕 Script Analyst Agent (the 7th)

Specialized for whole-script analysis: arc, pacing, structure, character. Different from the Script Agent (which generates new content) — this one *analyzes* what's there.

In [13]:
SCRIPT_ANALYST_SYSTEM = """You are an expert script consultant and story analyst.
You analyze screenplays for: character arc, three-act structure, pacing, dialogue voice, theme, and craft.
You give honest, specific notes — strengths AND weaknesses. You cite scene numbers and page-equivalents when possible.
You write in clear bullet points and numbered lists when listing notes."""

def script_analyst_agent(query: str, script_id: Optional[str] = None) -> str:
    if not script_id:
        return "Please upload a script first — I analyze whole screenplays for arc, pacing, structure, and character."

    data = load_script(script_id)
    if not data:
        return f"Script {script_id} not found."

    # Build a structural summary the LLM can reason over
    summary = f"""TITLE: {data['title']}
SCENE COUNT: {data['scene_count']}
CHARACTER COUNT: {data['character_count']}

SCENES OUTLINE:
""" + "\n".join([f"  Scene {s.number}: {s.heading} | chars: {', '.join(s.characters)}" for s in data['scenes']])

    # If the script is short, include full text. If long, use the structural summary + retrieved scenes.
    if len(data['content']) < 6000:
        body = f"\n\nFULL SCRIPT:\n{data['content']}"
    else:
        idx = get_script_index(script_id)
        relevant = idx.search(query, k=3) if idx else []
        if relevant:
            body = "\n\nMOST RELEVANT SCENES TO QUESTION:\n" + "\n\n".join([
                f"--- {s['heading']} ---\n{s['content'][:1500]}" for s, _ in relevant
            ])
        else:
            body = ""

    prompt = f"""{summary}{body}

DIRECTOR'S REQUEST: {query}

Provide concrete analysis. Cite scene numbers. Identify both strengths and revision opportunities.
"""
    return call_llm(prompt, system=SCRIPT_ANALYST_SYSTEM, agent_hint="script_analyst")

# Demo
print(script_analyst_agent("Analyze the protagonist's arc and pacing", script_id=test_id)[:500])


**Script Analysis: Maya's Arc**

**Want vs. Need:**
- Want: To uncover what her father hid
- Need: To accept that some truths can't be fully known

**Three-act structure:**
- Act 1 (pp 1-25): Returns home after father's death; finds first clue
- Act 2 (pp 26-75): Investigation deepens; lighthouse becomes obsession
- Act 3 (pp 76-95): Confronts the truth; lets go

**Pacing notes:**
- Strong opening hook (page 3)
- Sag in middle of Act 2 (pp 45-58) — consider compressing
- Climax lands cleanly

**


## 12. The 6 Agents (Now Script-Aware)

Same agents from v1, wrapped with `with_script_context`. When a `script_id` is passed, they automatically get relevant scenes injected.

In [14]:
# --- Script Agent (creative writing) ---
SCRIPT_SYSTEM = """You are an expert screenwriter. Help develop scripts, scenes, characters, dialogue.
Use proven frameworks: Save the Cat, Hero's Journey, 3-act."""
def _script_agent(query): return call_llm(query, system=SCRIPT_SYSTEM, agent_hint="script")
script_agent = with_script_context(_script_agent, "script")

# --- Music Agent (RAG) ---
MUSIC_SYSTEM = "You are a music supervisor for film. Recommend tracks matching scene mood."
def _music_agent(query):
    matches = music_store.search(query, k=5)
    candidates = "\n".join([f"- '{m['title']}' by {m['artist']} (mood: {m['mood']}, BPM {m['tempo']})" for m, _ in matches])
    prompt = f"{query}\n\nCandidates from library:\n{candidates}\n\nRecommend top 3-5 with reasoning."
    return call_llm(prompt, system=MUSIC_SYSTEM, agent_hint="music")
music_agent = with_script_context(_music_agent, "music")

# --- Camera Agent (RAG) ---
CAMERA_SYSTEM = "You are a veteran DP. Recommend camera movements, angles, lens choices."
def _camera_agent(query):
    matches = camera_store.search(query, k=4)
    knowledge = "\n".join([f"- {m['topic']}: {m['text']}" for m, _ in matches])
    prompt = f"{query}\n\nRelevant cinematography:\n{knowledge}\n\nRecommend movements and coverage."
    return call_llm(prompt, system=CAMERA_SYSTEM, agent_hint="camera")
camera_agent = with_script_context(_camera_agent, "camera")

# --- Tutor Agent (RAG, teaching mode) ---
TUTOR_SYSTEM = "You are a film school instructor. Teach step-by-step. Number every step."
def _tutor_agent(query):
    matches = camera_store.search(query, k=3)
    knowledge = "\n".join([f"- {m['topic']}: {m['text']}" for m, _ in matches])
    prompt = f"{query}\n\nReference techniques:\n{knowledge}\n\nTeach step-by-step. End with 'Common mistake'."
    return call_llm(prompt, system=TUTOR_SYSTEM, agent_hint="tutor")
tutor_agent = with_script_context(_tutor_agent, "tutor")

# --- Shot Idea Agent (creative) ---
SHOT_IDEA_SYSTEM = "You are a creative storyboard artist. Generate 6 distinct angle ideas per scene."
def _shot_idea_agent(query):
    return call_llm(query + "\n\nGenerate 6 distinct camera angles with emotional reasoning.",
                    system=SHOT_IDEA_SYSTEM, agent_hint="shot_ideas")
shot_idea_agent = with_script_context(_shot_idea_agent, "shot_ideas")

# --- Network Agent (RAG, no script context needed) ---
NETWORK_SYSTEM = "You are a film-industry matchmaker. Recommend producers/DPs/composers."
def network_agent(query, script_id=None):
    matches = network_store.search(query, k=3)
    profiles = "\n".join([f"- {m['name']} ({m['role']}): {m['specialty']}. Credits: {m['credits']}" for m, _ in matches])
    prompt = f"{query}\n\nCandidates:\n{profiles}\n\nRecommend top 3 with reasoning."
    return call_llm(prompt, system=NETWORK_SYSTEM, agent_hint="network")

print("✅ 7 agents defined (6 script-aware + 1 network)")
print("\nDemo: Music agent WITH script context:")
print(music_agent("Suggest music for the photograph reveal scene", script_id=test_id)[:300])


✅ 7 agents defined (6 script-aware + 1 network)

Demo: Music agent WITH script context:
Recommended tracks for this scene:

1. **'Time' — Hans Zimmer** — slow build, emotional weight (BPM 65)
2. **'An Ending (Ascent)' — Brian Eno** — ambient, contemplative
3. **'The Wolf' — SIAMÉS** — tense electronic (BPM 95)

Top pick: 'Time' — its slow piano-to-orchestral build matches a revelation 


## 13. LangGraph Orchestrator (v2)

Router now also considers whether a script is loaded — when it is, ambiguous queries lean toward `script_analyst`.

In [15]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END

class CinemaState(TypedDict):
    query: str
    intent: str
    response: str
    script_id: Optional[str]
    history: list

INTENT_KEYWORDS = {
    "script_analyst": [("analyze", 5), ("arc", 4), ("pacing", 5), ("structure", 4), ("critique", 5), ("review my script", 5), ("character development", 4)],
    "script":         [("write a scene", 5), ("script", 3), ("dialogue", 3), ("character arc", 4), ("plot", 3), ("outline", 3), ("screenplay", 4)],
    "music":          [("music", 5), ("song", 4), ("soundtrack", 5), ("track", 3), ("score", 3), ("composer", 4)],
    "camera":         [("camera movement", 5), ("dolly", 5), ("crane shot", 5), ("tracking shot", 5), ("lens", 4), ("lighting", 3), ("camera angle", 3)],
    "tutor":          [("how to", 4), ("how do i", 4), ("teach me", 5), ("explain", 3), ("tutorial", 5), ("step by step", 5), ("learn", 4)],
    "shot_ideas":     [("angle ideas", 5), ("shot ideas", 5), ("brainstorm", 4), ("framing", 4), ("composition", 4), ("angles for", 5)],
    "network":        [("producer", 5), ("find me", 4), ("collaborator", 5), ("hire", 4), ("cinematographer", 5)],
}

def route_intent(state: CinemaState) -> CinemaState:
    q = state["query"].lower()
    scores = {agent: sum(weight for kw, weight in kws if kw in q) for agent, kws in INTENT_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    if scores[best] == 0:
        # No keywords matched — if a script is loaded, default to analyst; else script
        state["intent"] = "script_analyst" if state.get("script_id") else "script"
    else:
        state["intent"] = best
    return state

def run_script(s):           s["response"] = script_agent(s["query"], script_id=s.get("script_id")); return s
def run_music(s):            s["response"] = music_agent(s["query"], script_id=s.get("script_id")); return s
def run_camera(s):           s["response"] = camera_agent(s["query"], script_id=s.get("script_id")); return s
def run_tutor(s):            s["response"] = tutor_agent(s["query"], script_id=s.get("script_id")); return s
def run_shot_ideas(s):       s["response"] = shot_idea_agent(s["query"], script_id=s.get("script_id")); return s
def run_network(s):          s["response"] = network_agent(s["query"]); return s
def run_script_analyst(s):   s["response"] = script_analyst_agent(s["query"], script_id=s.get("script_id")); return s

def select_agent(state) -> Literal["script","music","camera","tutor","shot_ideas","network","script_analyst"]:
    return state["intent"]

graph = StateGraph(CinemaState)
graph.add_node("router", route_intent)
for name, fn in [("script", run_script), ("music", run_music), ("camera", run_camera),
                 ("tutor", run_tutor), ("shot_ideas", run_shot_ideas),
                 ("network", run_network), ("script_analyst", run_script_analyst)]:
    graph.add_node(name, fn)

graph.set_entry_point("router")
graph.add_conditional_edges("router", select_agent, {a:a for a in ["script","music","camera","tutor","shot_ideas","network","script_analyst"]})
for a in ["script","music","camera","tutor","shot_ideas","network","script_analyst"]:
    graph.add_edge(a, END)

cinema_match_v2 = graph.compile()
print("✅ Orchestrator v2 compiled (7 agents)")

# Test
test_queries = [
    ("Analyze my script's pacing", test_id),
    ("Suggest music for the lighthouse scene", test_id),
    ("Find me a producer for indie thriller", None),
]
for q, sid in test_queries:
    r = cinema_match_v2.invoke({"query": q, "intent": "", "response": "", "script_id": sid, "history": []})
    print(f"\n🎬 {q}\n   Routed: {r['intent']} | Reply: {r['response'][:120]}...")


/Users/saiashish/cinema-match/.venv/lib/python3.11/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


✅ Orchestrator v2 compiled (7 agents)

🎬 Analyze my script's pacing
   Routed: script_analyst | Reply: **Script Analysis: Maya's Arc**

**Want vs. Need:**
- Want: To uncover what her father hid
- Need: To accept that some t...

🎬 Suggest music for the lighthouse scene
   Routed: music | Reply: Recommended tracks for this scene:

1. **'Time' — Hans Zimmer** — slow build, emotional weight (BPM 65)
2. **'An Ending ...

🎬 Find me a producer for indie thriller
   Routed: network | Reply: Top 3 matches:

1. **Priya Raman** — Producer, $500K-$2M indie thrillers. Score: 0.91
2. **Marcus Vela** — Cinematograph...


## 14. Gradio UI v2

New UI with:
- 📤 File upload component (any of the 5 supported formats)
- 💬 Chat with the script loaded into context
- 🔗 Share-link display (so users can bookmark/share)
- 📊 Script summary panel (scene count, characters)

The `gr.State` keeps the active `script_id` per-user-session in memory.

In [18]:
import gradio as gr

AGENT_LABELS = {
    "script": "📝 Script", "music": "🎵 Music", "camera": "🎥 Camera",
    "tutor": "🎓 Tutor", "shot_ideas": "💡 Shot Ideas",
    "network": "🤝 Network", "script_analyst": "📊 Script Analyst",
}

def handle_upload(file_obj, current_script_id):
    """Parse uploaded script, save to DB, return new script_id + summary."""
    if file_obj is None:
        return current_script_id, "No file uploaded.", ""

    try:
        text = parse_script(file_obj.name)
        scenes = extract_scenes(text)
        if not scenes:
            return current_script_id, "⚠️ No scenes detected. Make sure script uses INT./EXT. headers.", ""

        chars = extract_all_characters(scenes)
        title = Path(file_obj.name).stem
        sid = save_script(text, scenes, title=title)

        summary = f"""**📄 {title}**
- Scenes: {len(scenes)}
- Characters: {len(chars)}
- Top characters: {', '.join(chars[:5])}{"..." if len(chars) > 5 else ""}

**🔗 Share URL:** `{get_share_url(sid)}`

You can now ask Cinema Match about this script. Try: *"Analyze the protagonist's arc"* or *"Suggest music for scene 1"*.
"""
        return sid, summary, get_share_url(sid)
    except Exception as e:
        return current_script_id, f"❌ Parse error: {e}", ""

def chat_fn(message, history, script_id):
    """Run a query through the orchestrator."""
    result = cinema_match_v2.invoke({
        "query": message, "intent": "", "response": "",
        "script_id": script_id, "history": history,
    })
    label = AGENT_LABELS.get(result["intent"], "🎬")
    return f"**{label}**\n\n{result['response']}"

with gr.Blocks(title="🎬 Cinema Match v2", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎬 Cinema Match v2\n*AI assistant for film directors — now with script upload.*")

    script_id_state = gr.State(value=None)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📤 Upload Script")
            file_input = gr.File(
                label="Drop screenplay here",
                file_types=[".pdf", ".docx", ".txt", ".md", ".fountain"],
            )
            upload_btn = gr.Button("Analyze Script", variant="primary")
            summary_md = gr.Markdown("*No script uploaded yet. Cinema Match works without one too — try the chat.*")
            share_url = gr.Textbox(label="🔗 Share URL", interactive=False, visible=False)

        with gr.Column(scale=2):
            gr.Markdown("### 💬 Chat")
            chatbot = gr.Chatbot(height=500, type="messages")
            msg = gr.Textbox(placeholder="Ask about your script, music, camera, or anything film...", show_label=False)
            clear = gr.Button("Clear chat")

            gr.Examples(
                examples=[
                    "Analyze the protagonist's arc and pacing",
                    "Suggest music for the lighthouse reveal scene",
                    "What camera angles work for the photograph reveal?",
                    "Teach me how to shoot the dialogue between Maya and Elena",
                    "Brainstorm 6 angles for the lighthouse climb",
                    "Find a composer for atmospheric thrillers",
                ],
                inputs=msg,
            )

    def on_upload(file_obj, sid):
        new_sid, summary, url = handle_upload(file_obj, sid)
        return new_sid, summary, gr.update(value=url, visible=bool(url))
    upload_btn.click(on_upload, [file_input, script_id_state], [script_id_state, summary_md, share_url])

    def on_message(user_msg, history, sid):
        if not user_msg.strip():
            return history, ""
        reply = chat_fn(user_msg, history, sid)
        history = history + [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": reply},
        ]
        return history, ""
    msg.submit(on_message, [msg, chatbot, script_id_state], [chatbot, msg])
    clear.click(lambda: [], None, chatbot)

# Launch
from gradio import networking
networking.url_ok = lambda url: True

# Launch
demo.launch()

Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


## 15. Production Roadmap (updated for v2)

| Layer | v2 (this notebook) | Production v3 |
|---|---|---|
| Storage | SQLite, single-user | Postgres + Alembic migrations |
| File parsing | pypdf + python-docx | + OCR fallback for scanned PDFs (Tesseract) |
| Scene extraction | regex | + LLM-based fallback for non-standard formats |
| Per-script index | in-memory dict cache | Redis or pgvector |
| Sharing | URL-based, public | + optional password protection, expiry |
| Auth | none | Magic-link login (Supabase Auth) |
| UI | Gradio | Next.js + tRPC |
| Deploy | local notebook | HuggingFace Spaces (Gradio) → Fly.io (FastAPI) |
| Monitoring | print logs | LangSmith for traces, Sentry for errors |

### Quick wins to add next
1. **Real Claude:** `cfg.MOCK_MODE = False` + add `ANTHROPIC_API_KEY`
2. **Streaming responses:** `client.messages.stream(...)` for the long script analyst outputs
3. **Caching:** `functools.lru_cache` on `call_llm` keyed by (prompt, system) — saves $$ during dev
4. **Conversation memory:** populate `state.history` and pass as messages list to Claude
5. **Cost guard:** wrap `call_llm` with a daily token-budget cap
